In [ ]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [ ]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

MODEL = "gemini-3.8-flash"

In [ ]:
import json
from pydantic import BaseModel
from typing import List, Optional
from duckduckgo_search import DDGS

class BusinessProfile(BaseModel):
    business_type: str
    location: str
    business_structure: str
    operations: str
    scale: str
    online_operations: bool


class Requirement(BaseModel):
    name: str
    category: str
    what_is_it: str
    why_required: str
    applicability: str

    official_authority: Optional[str] = None
    official_portal: Optional[str] = None
    official_fee: Optional[str] = None

    required_documents: List[str] = []
    expected_timeline: Optional[str] = None

    can_apply_self: bool
    self_apply_explanation: str

    official_sources: List[str] = []
    third_party_sources: List[str] = []

    confidence: str

import time
import json

MODELS = [
    "gemini-3.8-flash"
]

def ask_gemini(prompt, max_retries=3):

    last_error = None

    for model in MODELS:

        for attempt in range(max_retries):

            try:

                print(
                    f"🤖 Using {model} "
                    f"(attempt {attempt + 1}/{max_retries})"
                )

                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config={
                        "temperature": 0,
                        "response_mime_type": "application/json"
                    }
                )

                return json.loads(response.text)

            except Exception as e:

                last_error = e

                print(
                    f"⚠️ {model} failed: {str(e)[:150]}"
                )

                # Exponential backoff
                time.sleep(2 ** attempt)

        print(
            f"➡️ Switching from {model}..."
        )

    raise RuntimeError(
        f"All Gemini models failed. Last error: {last_error}"
    )

def web_search(query, max_results=5):

    results = []

    with DDGS() as ddgs:

        for r in ddgs.text(
            query,
            max_results=max_results
        ):

            results.append({
                "title": r.get("title"),
                "url": r.get("href"),
                "snippet": r.get("body")
            })

    return results


def understand_business(user_input):

    prompt = f"""
You are LegalDoc's business intake agent.

LegalDoc helps first-time entrepreneurs in India understand
what registrations, licenses, permits and documents they may
need to start a business.

USER:

{user_input}

Extract the business information.

Return ONLY JSON:

{{
    "business_type": "",
    "location": "",
    "business_structure": "",
    "operations": "",
    "scale": "",
    "online_operations": true
}}

Rules:

- If something is unknown, use "unknown".
- Do not invent information.
- Infer only when clearly implied.
"""

    data = ask_gemini(prompt)

    return BusinessProfile(**data)



def find_requirements(profile):

    prompt = f"""
You are LegalDoc's compliance planning agent.

Business:

Type:
{profile.business_type}

Location:
{profile.location}

Structure:
{profile.business_structure}

Operations:
{profile.operations}

Scale:
{profile.scale}

Online:
{profile.online_operations}

Identify registrations, licenses, permits and important
compliance documents that should be investigated.

Consider:

- Central government
- State government
- Municipal/local authorities
- Tax
- Sector-specific licenses
- Premises
- Employees
- Online operations
- Business structure

Do NOT assume every item is mandatory.

Return ONLY JSON:

[
    {{
        "name": "",
        "category": "",
        "reason": ""
    }}
]
"""

    return ask_gemini(prompt)
def search_requirement(requirement, profile):

    query = (
        f'"{requirement["name"]}" '
        f'{profile.location} '
        f'official government India'
    )

    print(f"   🔎 Searching: {query}")

    results = web_search(
        query,
        max_results=5
    )

    # Remove duplicate URLs
    unique = {}

    for result in results:

        url = result.get("url")

        if url:
            unique[url] = result

    print(f"   📄 Found {len(unique)} sources")

    return list(unique.values())

def research_requirement(
    requirement,
    profile,
    sources
):

    source_text = ""

    for source in sources:

        source_text += f"""
TITLE:
{source['title']}

URL:
{source['url']}

DESCRIPTION:
{source['snippet']}

--------------------------------
"""

    prompt = f"""
You are LegalDoc's compliance research agent.

Research the following requirement for this business.

BUSINESS

Type:
{profile.business_type}

Location:
{profile.location}

Structure:
{profile.business_structure}

Operations:
{profile.operations}

Scale:
{profile.scale}


REQUIREMENT

Name:
{requirement['name']}

Category:
{requirement['category']}

Reason:
{requirement['reason']}


SEARCH RESULTS

{source_text}


IMPORTANT RULES

1. Prefer official government sources.

2. Government sources generally include:
   *.gov.in
   *.nic.in
   official municipal/state government domains.

3. Never call a private CA/consultant website
   an official source.

4. Never invent fees.

5. Never invent timelines.

6. If something cannot be verified, write:
   "Not verified"

7. Determine whether the requirement actually applies
   to this specific business.

8. Explain conditions such as turnover, location,
   business type or premises.

9. Separate official sources from third-party sources.

10. This is informational guidance, not legal advice.

Return ONLY JSON:

{{
    "name": "",
    "category": "",
    "what_is_it": "",
    "why_required": "",
    "applicability": "",

    "official_authority": "",
    "official_portal": "",

    "official_fee": "",

    "required_documents": [],

    "expected_timeline": "",

    "can_apply_self": true,

    "self_apply_explanation": "",

    "official_sources": [],

    "third_party_sources": [],

    "confidence": "High"
}}
"""

    data = ask_gemini(prompt)

    return Requirement(**data)

def LegalDoc(user_input):

    print(" Understanding your business...")

    profile = understand_business(user_input)

    print("\nBusiness profile:")
    print(profile.model_dump_json(indent=2))


    print("\n Identifying possible requirements...")

    requirements = find_requirements(profile)

    print(
        f"Found {len(requirements)} requirements to investigate."
    )


    researched = []


    for i, requirement in enumerate(requirements):

        print(
            f"\n Researching "
            f"{i+1}/{len(requirements)}: "
            f"{requirement['name']}"
        )

        sources = search_requirement(
            requirement,
            profile
        )

        try:

            result = research_requirement(
                requirement,
                profile,
                sources
            )

            researched.append(result)

        except Exception as e:

            print(
                f"⚠️ Could not research "
                f"{requirement['name']}: {e}"
            )


    return {
        "business_profile": profile.model_dump(),
        "requirements": [
            r.model_dump()
            for r in researched
        ]
    }

In [ ]:
user_input = """
I want to start a small momo and Chinese food stall
in Andheri West, Mumbai.

It will be a sole proprietorship.
I will rent a small shop.
I will sell directly to customers and through
Swiggy and Zomato.

My expected annual turnover is around 8 lakh.
"""

result = LegalDoc(user_input)

In [ ]:
print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
!pip -q install -U google-genai

In [ ]:
import os
from getpass import getpass
from google import genai

GEMINI_API_KEY = getpass("Enter Gemini API key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

for model in client.models.list():
    print(model.name)